In [1]:
import  torch

In [2]:
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    recall_score,
    confusion_matrix
)


import numpy as np 
import pandas as pd 

In [3]:
!pip install -U transformers accelerate safetensors
#safetensors->model.safetensors (new format)

Le****t's load the dataset****

In [4]:
from datasets import load_dataset

dataset = load_dataset("go_emotions")

In [5]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 43410
    })
    validation: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 5426
    })
    test: Dataset({
        features: ['text', 'labels', 'id'],
        num_rows: 5427
    })
})

In [6]:
dataset['train'][0]

{'text': "My favourite food is anything I didn't have to cook myself.",
 'labels': [27],
 'id': 'eebbqej'}

In [7]:
dataset['train']['labels'][0:10]

[[27], [27], [2], [14], [3], [26], [15], [8, 20], [0], [27]]

In [8]:
label_names = dataset['train'].features['labels'].feature.names
label_names

['admiration',
 'amusement',
 'anger',
 'annoyance',
 'approval',
 'caring',
 'confusion',
 'curiosity',
 'desire',
 'disappointment',
 'disapproval',
 'disgust',
 'embarrassment',
 'excitement',
 'fear',
 'gratitude',
 'grief',
 'joy',
 'love',
 'nervousness',
 'optimism',
 'pride',
 'realization',
 'relief',
 'remorse',
 'sadness',
 'surprise',
 'neutral']

In [9]:
dataset = dataset.remove_columns(['id'])

In [10]:
dataset

DatasetDict({
    train: Dataset({
        features: ['text', 'labels'],
        num_rows: 43410
    })
    validation: Dataset({
        features: ['text', 'labels'],
        num_rows: 5426
    })
    test: Dataset({
        features: ['text', 'labels'],
        num_rows: 5427
    })
})

In [11]:
train_ds=dataset['train']
test_ds= dataset['test']
val_ds= dataset['validation']

In [12]:
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

Now, we are gonna tokenize our texts.From words to token IDS.For that purpose, we use Autotokenizer/BertTokenizer

In [13]:
def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

In [14]:
train_ds= train_ds.map(tokenize, batched=True)

In [15]:
test_ds= test_ds.map(tokenize, batched=True)
val_ds= val_ds.map(tokenize, batched=True)

### BERT Model for Sequence Classification

A pre-trained BERT model is loaded with an added classification head.
The number of output labels is set to match the number of emotion classes.

This allows the model to be fine-tuned specifically for emotion detection.


In [16]:
train_ds

Dataset({
    features: ['text', 'labels', 'input_ids', 'token_type_ids', 'attention_mask'],
    num_rows: 43410
})

In [17]:
train_ds = train_ds.remove_columns(["text", "token_type_ids"])
test_ds = test_ds.remove_columns(["text", "token_type_ids"])
val_ds = val_ds.remove_columns(["text", "token_type_ids"])

dataset clearly represents a multi-label emotion classification problem. ->[0,8]
-->ValueError: expected sequence of length 1 at dim 1 (got 2)

So, Let's convert the labels into multi-hot vectors and configure the model with problem_type="multi_label_classification".

In [18]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()

train_multi_hot_labels = mlb.fit_transform(train_ds['labels'])
test_multi_hot_labels = mlb.fit_transform(test_ds['labels'])
val_multi_hot_labels = mlb.fit_transform(val_ds['labels'])

In [19]:
train_multi_hot_labels[7:8]

array([[0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0,
        0, 0, 0, 0, 0, 0]])

In [20]:
from transformers import AutoModelForSequenceClassification

Loads pretrained BERT weights

Adds a classification head on top

Output size = number of emotions (6 in your case)

📌 The label2id / id2label part is for readable predictions, not training logic.

In [21]:
model=AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=28,
    problem_type="multi_label_classification"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [22]:
train_ds = train_ds.remove_columns(["labels"])

In [23]:
train_ds

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 43410
})

In [24]:
labels = torch.tensor(train_multi_hot_labels).float()

In [25]:
labels_list = train_multi_hot_labels.tolist()

In [26]:
train_ds = train_ds.add_column("labels", labels_list)

In [27]:
train_ds

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 43410
})

In [28]:
train_ds['labels']

Column([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1], [0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], [0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], ...])

In [29]:
print(train_ds.features)

{'input_ids': List(Value('int32')), 'attention_mask': List(Value('int8')), 'labels': List(Value('int64'))}


In [30]:
from datasets import Value, Sequence

# Update the dataset to ensure labels are float32
train_ds = train_ds.cast_column(
    "labels",
    Sequence(Value("float32"))
)

In [31]:
print(train_ds.features)

{'input_ids': List(Value('int32')), 'attention_mask': List(Value('int8')), 'labels': List(Value('float32'))}


In [32]:
train_ds.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

In [33]:
model.config.id2label = {i: label for i, label in enumerate(label_names)}
model.config.label2id = {label: i for i, label in enumerate(label_names)}

In [34]:
!pip uninstall -y peft

In [ ]:
from transformers import Trainer, TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=8,
    num_train_epochs=3,
    learning_rate=2e-5,
    logging_steps=100,
    save_strategy="epoch",
)

In [63]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    compute_metrics=compute_metrics
)

In [36]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Step,Training Loss
100,0.663364
200,0.326555
300,0.296581
400,0.283970
500,0.267725
600,0.261317
700,0.244397
800,0.238200
900,0.235033
1000,0.220010


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=8142, training_loss=0.18388331179477108, metrics={'train_runtime': 2307.897, 'train_samples_per_second': 56.428, 'train_steps_per_second': 3.528, 'total_flos': 8568237917583360.0, 'train_loss': 0.18388331179477108, 'epoch': 3.0})

In [38]:
test_ds

Dataset({
    features: ['labels', 'input_ids', 'attention_mask'],
    num_rows: 5427
})

In [41]:
test_ds = test_ds.remove_columns(["labels"])

In [40]:
labels_list_test = test_multi_hot_labels.tolist()

In [42]:
test_ds = test_ds.add_column("labels", labels_list_test)

In [44]:
test_ds = test_ds.cast_column(
    "labels",
    Sequence(Value("float32"))
)

Casting the dataset:   0%|          | 0/5427 [00:00<?, ? examples/s]

In [45]:
test_ds.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

In [46]:
test_ds

Dataset({
    features: ['input_ids', 'attention_mask', 'labels'],
    num_rows: 5427
})

In [64]:
from transformers.utils.notebook import NotebookProgressCallback
trainer.remove_callback(NotebookProgressCallback)

In [56]:
from sklearn.metrics import precision_score

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = 1 / (1 + np.exp(-logits))  # Sigmoid
    preds = (probs > 0.3).astype(int)

    return {
        "micro_f1": f1_score(labels, preds, average="micro", zero_division=0),
        "macro_f1": f1_score(labels, preds, average="macro", zero_division=0),
        "precision": precision_score(labels, preds, average="micro", zero_division=0),
        "recall": recall_score(labels, preds, average="micro", zero_division=0),
    }

In [66]:
metrics = trainer.evaluate(test_ds)
print(metrics)

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


{'eval_loss': 0.16873039305210114, 'eval_micro_f1': 0.6060044763448329, 'eval_macro_f1': 0.46566095473235997, 'eval_precision': 0.5923355461677731, 'eval_recall': 0.6203191657449834, 'eval_runtime': 31.2598, 'eval_samples_per_second': 173.61, 'eval_steps_per_second': 10.877, 'epoch': 0}


In [55]:
predictions = trainer.predict(test_ds)
logits = predictions.predictions
labels = predictions.label_ids

probs = 1 / (1 + np.exp(-logits))

best_threshold = 0.5
best_f1 = 0

for threshold in np.arange(0.1, 0.9, 0.05):
    preds = (probs > threshold).astype(int)
    f1 = f1_score(labels, preds, average="micro")
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = threshold

print("Best Threshold:", best_threshold)
print("Best Micro F1:", best_f1)

/usr/local/lib/python3.12/dist-packages/torch/nn/parallel/_functions.py:71: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  warnings.warn(


Best Threshold: 0.30000000000000004
Best Micro F1: 0.6060044763448329


In [69]:
trainer.save_model("emotion_bert_model")
tokenizer.save_pretrained("emotion_bert_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('emotion_bert_model/tokenizer_config.json',
 'emotion_bert_model/tokenizer.json')